# Recommendation Systems

Recommendation systems are used to suggest products to users based on their past preferences. Recommendation systems are generally divided into the following categories:

- **Content-based**: Based on the similarity between product features (e.g. keywords, categories) and user preferences.  
  For example, if a user liked Product A, the recommendation system suggests products with similar features. In the context of movies, these features could be director, genre, etc. This approach can typically be implemented with a supervised binary classification algorithm.

- **Collaborative filtering**: Computes similarity from user–item interactions (e.g. ratings, purchase counts, likes, etc.).  
  This method finds customers with similar preferences and then recommends items they haven't experienced yet but that similar customers have preferred. The system assumes that users with similar movie-watching habits generally have similar tastes. It finds other users who watch videos similar to what the target user watches, then recommends videos those users have watched but the target user hasn't. There are 3 types: user–user collaborative filtering, item–item collaborative filtering, and matrix factorization.

- Newer and generally more powerful approaches are **Hybrid systems**.  
  These systems combine the two methods mentioned above.

<img src="https://wagon-public-datasets.s3.amazonaws.com/data-science-images/05-ML/06-Unsupervised-Learning/recommendation_systems.png" width=600>

The diagram above provides a detailed classification of the different methodologies used to build a recommendation system.

In the context of `movieLens`, which will be used as a sample dataset later:
- We can recommend different movies based on content similarity (e.g. genre, cast, etc.); this applies **item-content filtering**.
- By comparing user metadata (e.g. age, gender), we could recommend items liked by similar users; this would use **user-content filtering**. However, since the `movieLens` dataset does not contain user content data, we will only build **item-item collaborative filtering**.

**Memory-based content filtering**

In memory-based methods, there is no model that learns from data and makes predictions. Instead, a **pre-computed similarity matrix** is created that can be used for movie recommendations.

## Data Collection and Cleaning

Run the lines below to download the required datasets. Then load the datasets into three separate pandas DataFrames (`movies`, `tags`, and `ratings`).

In [ ]:
!curl https://d32aokrjazspmn.cloudfront.net/materials/movie_ratings.csv > data/movies.csv
!curl https://d32aokrjazspmn.cloudfront.net/materials/movie_tags.csv > data/tags.csv
!curl https://d32aokrjazspmn.cloudfront.net/materials/movie_titles.csv > data/ratings.csv

In [ ]:
# YOUR CODE HERE

__Remove the '|' character that separates different movie genres and replace it with a space.__

In [ ]:
# YOUR CODE HERE

__Filter the `movies` DataFrame to keep only movies that have received a rating.__

In [ ]:
# YOUR CODE HERE

### 🧪 Test Your Code

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'datasets',
    movies_shape=movies.shape,
    tags_shape=tags.shape,
    ratings_shape=ratings.shape,
    genres_cleaned=movies['genres'].str.find('|') >= 0
)

result.write()
print(result.check())

## Feature Engineering

We will create a new feature called `metadata` that combines **all the text data** we have about a movie — its genres and tags.

__Merge the `movies` and `tags` DataFrames.__

In [ ]:
# YOUR CODE HERE

**Create a new `merged_df` DataFrame containing a `metadata` column. This column should be formed by concatenating both the tags and the genres.**

To do this:
- First, **aggregate the tags** per movie.
- Then, concatenate those aggregated tags with the **genres** column.

For example, the `metadata` column for the movie *Toy Story* should look like:  
`pixar pixar fun Adventure Animation Children Comedy Fantasy`

👉 `merged_df` should contain at least the following columns: `movieId`, `title`, and `metadata`.

In [ ]:
# YOUR CODE HERE

### 🧪 Test Your Code

In [ ]:
from nbresult import ChallengeResult
import numpy as np

result = ChallengeResult(
    'feature_engineering',
    unique_movies=np.all(merged_df[['movieId']].value_counts() > 1),
    metadata=merged_df[merged_df['title'] == 'Copycat (1995)'],
    merged_df_rows=merged_df.shape[0]
)

result.write()
print(result.check())

## Building a Content Latent Matrix from Metadata

### Count Vectorizer

In the next step, we need to convert the metadata texts into vectors so we can feed them into machine learning algorithms. Machine learning models cannot directly understand text data, so we need to encode it.

For this purpose, we will encode the `metadata` column using [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html).

Create a new `count_df` DataFrame from the vectors obtained after the count transformation.  
Each row will represent the **frequency vector** of the corresponding movie.

In [ ]:
# YOUR CODE HERE

### Dimensionality Reduction

Each movie's metadata was converted into a vector of approximately **1675 dimensions**!

As we saw in previous lessons, we can apply **dimensionality reduction** methods to represent the data (movies) with minimal information loss. **Truncated Singular Value Decomposition (SVD)** is another advanced tool used for dimensionality reduction.

Unlike PCA, this method **does not center the data before computing the singular value decomposition (SVD)**. This makes it capable of **working efficiently with sparse matrices**. It works particularly well on term count / frequency matrices. In this context, it is also known as **Latent Semantic Analysis (LSA)**.

You can look at the [`TruncatedSVD`](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html) class in Scikit-Learn; the principle is the same as PCA.

__👉 **Apply Truncated SVD to reduce the dimensionality of your count matrix (e.g. to 25 features).**__


In [ ]:
# YOUR CODE HERE

👉 **Plot the total explained variance ratio as a function of the number of components.**

In [ ]:
# YOUR CODE HERE

With only the first 25 components (out of ~1700 originally), we can explain **more than 80%** of the variance, which is sufficient for our purposes.

👉 **Save the 25 components of this matrix into a new `latent_df` DataFrame indexed by movie titles.**

In [ ]:
# YOUR CODE HERE

### 🧪 Test Your Code

In [ ]:
from nbresult import ChallengeResult

result =  ChallengeResult(
    'metadata',
    counter_shape=count_df.shape,
    latent_shape=latent_df.shape
)

result.write()
print(result.check())

## Building a Latent Matrix from User Ratings

Beyond metadata, we have another valuable source of information: **user ratings**.

A recommendation system can suggest a similar movie based on user ratings (item-to-item collaborative filtering).

👉 **We prepare the following dataset with movies as rows and `userId`s as columns.**

In [ ]:
# Merge
ratings1 = pd.merge(movies[['movieId']], ratings, on="movieId", how="right")
# Pivot
ratings2 = ratings1.pivot(index = 'movieId', columns ='userId', values = 'rating').fillna(0)
display(ratings2.head())
ratings2.shape

We created a dataset containing user ratings as vectors of length 9724.

👉 **Again, we will apply SVD to the `ratings2` DataFrame, keeping the first 200 components. Name this DataFrame `latent_df_2`.**

In [ ]:
# YOUR CODE HERE

**👉 Re-index by Movie Title.**

In [ ]:
# YOUR CODE HERE

In [ ]:
latent_df_2.shape

### 🧪 Test Your Code

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult('ratings', latent_shape=latent_df_2.shape)

result.write()
print(result.check())

## Applying Cosine Similarity on Content and Collaborative Matrices

In the next step, we will use a similarity measure to find the **top $ movies most similar to "Toy Story"** based on the filtering methods we built. One similarity measure we can use is **Cosine Similarity**. Scikit-learn provides the [`cosine_similarity`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html) function for this.

👉 **Compute the cosine similarity for a sample movie ("Toy Story") against both the content (metadata) and collaborative (ratings) latent matrices.**

In [ ]:
# YOUR CODE HERE

We can also build a hybrid filter that averages the similarity of content and collaborative filtering.

`hybrid_similarity = (content_similarity + collaborative_similarity) / 2`

__Compute the average similarity of content and collaborative filtering.__

In [ ]:
# YOUR CODE HERE

__Create a DataFrame containing the final similarities with Toy Story.__

In [ ]:
# YOUR CODE HERE

__Sort your DataFrame by the most similar movies in terms of collaborative similarity.__

In [ ]:
# YOUR CODE HERE

Of course, you should see Toy Story as the most similar movie (similarity of 1 for each column).

__You can sort by content and hybrid type and see which one gives the best recommendation.__

In [ ]:
# YOUR CODE HERE

**❓ Which similarity is better for building a realistic movie recommendation system? Assign it to the `best_similarity` variable.**

In [ ]:
a = 'content'
b = 'collaborative'
c = 'hybrid'
best_similarity = None # fill in with the right answer

### 🧪 Test Your Code

In [ ]:
from nbresult import ChallengeResult

result = ChallengeResult('recommender', best_similarity=best_similarity)

result.write()
print(result.check())

## 🏁 Well Done!

You can commit and push your code to GitHub.